# Permutations and Combinations

How many unique 3-character passwords can you make from `A B C D E`, without reusing a character?

A programmer's instinct: write a loop and count. That instinct is correct — and it is exactly what the mathematics is doing. Every formula in this notebook is a compact description of a loop you would write instinctively.

**Prerequisites:** Notebooks 00–07. Comfort with loops and multiplication.

In [ ]:
from itertools import permutations

chars = ['A', 'B', 'C', 'D', 'E']
passwords = list(permutations(chars, 3))

print(f"Unique 3-character passwords from {chars}, no repeats: {len(passwords)}")
print("\nFirst 15:")
for i, p in enumerate(passwords[:15]):
    print(f"  {''.join(p)}", end="  ")
    if (i + 1) % 5 == 0:
        print()

60 passwords. Two concepts cover all counting problems of this type:

- **Permutations** — arrangements where order matters. `ABC` and `BAC` are different.
- **Combinations** — selections where order does not matter. `ABC` and `BAC` are the same group.

Both come in two variants: with repetition and without. Four cases total, all built from scratch.

---

## 1. Factorial

The factorial of `n`, written `n!`, is the product of every integer from `n` down to 1:

$$5! = 5 \times 4 \times 3 \times 2 \times 1 = 120$$

That is a loop — a countdown multiply. Factorial appears throughout counting because arranging objects means multiplying a shrinking pool of choices, and that shrinking product is the factorial loop.

In [ ]:
def factorial(n):
    result = 1
    for i in range(n, 0, -1):  # count down: n, n-1, ..., 1
        result *= i
    return result

print("Building 5! step by step:")
running = 1
for i in range(5, 0, -1):
    running *= i
    print(f"  x {i}  =>  {running}")

import math
print()
print("Values for n = 0 through 9:")
for n in range(10):
    print(f"  {n}! = {factorial(n):8,}   verified: {factorial(n) == math.factorial(n)}")

---

## 2. Permutations Without Repetition — All Objects

Arrange all 5 characters in a sequence. How many orderings exist?

Fill 5 positions left to right. After each placement the pool shrinks:

| Position | Available choices |
|----------|-------------------|
| 1 | 5 |
| 2 | 4 (one used) |
| 3 | 3 (two used) |
| 4 | 2 (three used) |
| 5 | 1 (four used) |

Total: $5 \times 4 \times 3 \times 2 \times 1 = 5! = 120$.

The full factorial counts every way to drain the entire pool.

In [ ]:
from itertools import permutations

chars = ['A', 'B', 'C', 'D', 'E']
all_orderings = list(permutations(chars))  # no r argument => use all

print(f"All orderings of {len(chars)} characters:")
print(f"  itertools count: {len(all_orderings)}")
print(f"  5! =             {factorial(5)}")
print(f"  Match:           {len(all_orderings) == factorial(5)}")
print()
print("Pool shrinking, step by step:")
product = 1
for pos in range(len(chars)):
    choices = len(chars) - pos
    product *= choices
    print(f"  Position {pos+1}: {choices} choices  =>  running product = {product}")

---

## 3. Permutations Without Repetition — r from n  ★

Now the core case: choose and arrange **r** objects from a pool of **n**, without reusing. Order matters.

**Example:** 3-character passwords from 5 characters, no repeats.

The programmer's natural solution: the pool starts at 5, shrinks by 1 after each pick, and you stop after 3 picks. That is a bounded loop — multiply down from `n`, stop after `r` steps.

In [ ]:
def perm_loop(n, r):
    """P(n, r) — the programmer's natural solution.
    Multiply n down, stopping after r steps.
    """
    result = 1
    for i in range(n, n - r, -1):   # range(5, 2, -1) => [5, 4, 3]
        result *= i
    return result

n, r = 5, 3
print(f"P({n},{r}): {r} items from a pool of {n}, no repeats, order matters")
print()
print(f"Loop iterates: range({n}, {n-r}, -1) = {list(range(n, n-r, -1))}")
print()
running = 1
for step, i in enumerate(range(n, n - r, -1)):
    running *= i
    print(f"  Step {step+1}: {i} choices remaining  =>  running product = {running}")
print()
print(f"Result: P({n},{r}) = {perm_loop(n, r)}")

The loop gives 60. Now here is the standard formula:

$$P(n, r) = \frac{n!}{(n-r)!}$$

At first glance this looks different — compute two factorials and divide. Expand it for $n=5$, $r=3$ and watch what happens:

$$P(5,3) = \frac{5!}{(5-3)!} = \frac{5!}{2!} = \frac{5 \times 4 \times 3 \times 2 \times 1}{2 \times 1}$$

The denominator `2 × 1` is exactly the tail of `5!` that the bounded loop never multiplied. It cancels:

$$= \frac{5 \times 4 \times 3 \times \cancel{2 \times 1}}{\cancel{2 \times 1}} = 5 \times 4 \times 3 = 60$$

**The division was never really there.** The formula is a compact encoding of the bounded loop: compute the full factorial, then cancel the tail you did not want. The loop just avoids creating the tail in the first place.

The formula and the loop are the same thing. The formula is code to *write*, not to *run* — and when you write it out, it reveals the loop.

In [ ]:
from math import factorial as math_factorial

n, r = 5, 3

numerator_terms   = list(range(n, 0, -1))      # [5, 4, 3, 2, 1]
denominator_terms = list(range(n - r, 0, -1))  # [2, 1]
surviving_terms   = numerator_terms[:r]         # [5, 4, 3]  <- what the loop computes
cancelled_terms   = numerator_terms[r:]         # [2, 1]  <- what the denominator cancels

print(f"Expanding P({n},{r}) = {n}! / {n-r}!")
print()
print(f"  Numerator  {n}!:   {' x '.join(str(x) for x in numerator_terms)}  = {math_factorial(n)}")
print(f"  Denominator {n-r}!:  {' x '.join(str(x) for x in denominator_terms)}  = {math_factorial(n-r)}")
print()

# Mark cancelled terms with brackets
marked = [str(x) for x in surviving_terms] + [f"[{x}]" for x in cancelled_terms]
print(f"  After cancellation:")
print(f"    Numerator:   {' x '.join(marked)}")
print(f"    Denominator: {' x '.join(f'[{x}]' for x in cancelled_terms)}")
print(f"    (bracketed terms cancel)")
print()
print(f"  Remaining: {' x '.join(str(x) for x in surviving_terms)} = {perm_loop(n, r)}")
print()
print(f"  Loop result:    {perm_loop(n, r)}")
print(f"  Formula result: {math_factorial(n) // math_factorial(n - r)}")
print(f"  Same?           {perm_loop(n, r) == math_factorial(n) // math_factorial(n - r)}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.style.use('dark_background')

n, r = 5, 3
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Left: pool shrinking at each position ---
ax = axes[0]
pool_sizes = [n - i for i in range(r)]
bar_colors = ['#4c9be8', '#56b4a0', '#e8914c']
bars = ax.bar(range(r), pool_sizes, color=bar_colors, edgecolor='#666666',
              linewidth=0.8, width=0.55)
ax.set_xticks(range(r), labels=[f'Position {i+1}' for i in range(r)], fontsize=11)
ax.set_ylabel('Available choices', fontsize=11)
ax.set_ylim(0, n + 1.5)
ax.set_title(f'P({n},{r}): the pool shrinks at each position', fontsize=12)
ax.grid(axis='y', alpha=0.2)
for bar, size in zip(bars, pool_sizes):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
            str(size), ha='center', va='bottom', fontsize=22, fontweight='bold')
product_str = ' x '.join(str(s) for s in pool_sizes)
ax.text(0.5, 0.07, f'{product_str} = {perm_loop(n, r)}',
        transform=ax.transAxes, ha='center', fontsize=13,
        color='#ffd700', fontweight='bold')

# --- Right: P(n,r) for a range of inputs ---
ax = axes[1]
pairs = [(3,1),(3,2),(3,3),(4,2),(4,3),(5,2),(5,3),(5,4),(6,2),(6,3)]
labels = [f'P({a},{b})' for a, b in pairs]
values = [perm_loop(a, b) for a, b in pairs]
bar_cols = plt.cm.plasma(np.linspace(0.15, 0.85, len(pairs)))
hbars = ax.barh(labels, values, color=bar_cols, edgecolor='#555555', linewidth=0.5)
for hbar, val in zip(hbars, values):
    ax.text(hbar.get_width() + max(values) * 0.01,
            hbar.get_y() + hbar.get_height() / 2,
            str(val), va='center', fontsize=9, color='white')
ax.set_xlabel('Number of permutations', fontsize=11)
ax.set_title('P(n,r) without repetition', fontsize=12)
ax.grid(axis='x', alpha=0.2)

plt.tight_layout()
plt.show()

---

## 4. Permutations With Repetition

What if characters can repeat? `AAB` is now a valid password.

The pool no longer shrinks. At every position, all `n` characters are available regardless of what came before — each position is an independent choice:

$$n^r$$

For 3 characters from 5, with repetition: $5^3 = 125$.

Compare: 125 with repetition vs 60 without. The extra 65 passwords all contain at least one repeated character.

This is the simpler case — it does not require factorial at all, because there is no shrinking pool to track.

In [ ]:
from itertools import product as cartesian_product

chars = ['A', 'B', 'C', 'D', 'E']
n, r = len(chars), 3

passwords_with_rep = list(cartesian_product(chars, repeat=r))

print(f"Permutations WITH repetition: n^r = {n}^{r} = {n**r}")
print(f"itertools.product count:             {len(passwords_with_rep)}")
print(f"Match: {len(passwords_with_rep) == n**r}")
print()
print(f"Without repetition: P({n},{r}) = {perm_loop(n, r)}")
print(f"With repetition:    {n}^{r}     = {n**r}")
print(f"Difference:                     {n**r - perm_loop(n, r)} extra passwords (contain at least one repeat)")
print()
# Show examples with repeated characters
repeated = [''.join(p) for p in passwords_with_rep if len(set(p)) < r]
print(f"Examples with repetition (first 10 of {len(repeated)}):")
print('  ' + '  '.join(repeated[:10]))

---

## 5. Combinations Without Repetition

Permutations count ordered arrangements. Combinations count unordered selections — the group matters, not the sequence within it.

`ABC`, `ACB`, `BAC`, `BCA`, `CAB`, `CBA` are 6 different permutations of the same 3 characters. As a combination, all six are the same selection: `{A, B, C}`.

To go from permutations to combinations, divide out the overcounting. Each group of `r` distinct characters has exactly `r!` internal orderings. Dividing by `r!` collapses all those orderings into one:

$$C(n, r) = \frac{P(n, r)}{r!} = \frac{n!}{r! \cdot (n-r)!}$$

For $n=5$, $r=3$:

$$C(5,3) = \frac{P(5,3)}{3!} = \frac{60}{6} = 10$$

Same pool, same size of selection — but now `ABC` and `BAC` count as one.

In [ ]:
from itertools import combinations
from math import factorial as math_factorial

def comb(n, r):
    return perm_loop(n, r) // math_factorial(r)

n, r = 5, 3
all_combos = list(combinations(['A','B','C','D','E'], r))

print(f"C({n},{r}): selections of {r} from {n}, order irrelevant")
print()
print(f"  P({n},{r})  = {perm_loop(n,r)}  (ordered arrangements, section 3)")
print(f"  r!   = {r}! = {math_factorial(r)}  (orderings within each group)")
print(f"  C({n},{r}) = {perm_loop(n,r)} / {math_factorial(r)} = {comb(n,r)}")
print()
print(f"  itertools.combinations count: {len(all_combos)}")
print(f"  Match: {len(all_combos) == comb(n,r)}")
print()
print(f"All {comb(n,r)} combinations:")
for i, combo in enumerate(all_combos):
    print(f"  {''.join(combo)}", end="  ")
    if (i + 1) % 5 == 0:
        print()

In [ ]:
# Show why we divide by r!: every group has exactly r! orderings to collapse
from itertools import permutations as all_perms

group = ('A', 'B', 'C')
orderings = list(all_perms(group))

print(f"All {len(orderings)} orderings of group {group}:")
for p in orderings:
    print(f"  {''.join(p)}", end="")
print()
print()
print(f"As a combination all {len(orderings)} reduce to one: {{{','.join(group)}}}")
print(f"Dividing P(5,3)={perm_loop(5,3)} by {len(orderings)} (= {len(group)}!) collapses every group.")
print()
# Confirm for all 10 combinations: each has exactly 3! = 6 permutations
from itertools import combinations, permutations
all_combos = list(combinations(['A','B','C','D','E'], 3))
print("Each combination has exactly 3! = 6 permutations:")
for combo in all_combos[:3]:
    count = len(list(permutations(combo)))
    print(f"  {combo}: {count} orderings")
print(f"  ... (same for all {len(all_combos)})")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from math import factorial as math_factorial

plt.style.use('dark_background')

n, r = 5, 3

cases = [
    ('Perm\nwith rep\n' + f'n^r',    n**r,             '#e8914c'),
    ('Perm\nwithout rep\n' + f'P(n,r)', perm_loop(n,r), '#4c9be8'),
    ('Perm\nall objects\n' + f'n!',  math_factorial(n), '#a78bde'),
    ('Comb\nwithout rep\n' + f'C(n,r)', comb(n,r),      '#56b4a0'),
]

labels = [c[0] for c in cases]
values = [c[1] for c in cases]
colors = [c[2] for c in cases]

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(range(len(cases)), values, color=colors, edgecolor='#666666',
              linewidth=0.8, width=0.55)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            str(val), ha='center', va='bottom', fontsize=18, fontweight='bold')
ax.set_xticks(range(len(cases)))
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylabel('Count', fontsize=12)
ax.set_ylim(0, max(values) * 1.3)
ax.set_title(f'All four counting methods compared — n={n}, r={r}', fontsize=13)
ax.grid(axis='y', alpha=0.2)
ax.text(0.5, 0.93,
        f'Same pool: {n} characters   |   Choosing {r}   |   Order matters -> permutations   |   Order irrelevant -> combinations',
        transform=ax.transAxes, ha='center', fontsize=8.5, color='#aaaaaa', style='italic')
plt.tight_layout()
plt.show()

---

## Exercises

1. A PIN uses 4 digits from `0–9`. How many unique PINs are there **with** repetition? How many **without**?
2. How many ways can 8 runners finish a race (first, second, third place only)? Which formula applies?
3. A committee of 3 is chosen from 8 candidates. Does order matter here? Compute the number of possible committees.
4. Use `comb(n, r)` to compute `C(10, 3)` and `C(10, 7)`. What do you notice? Explain why using the formula.
5. The cancellation in section 3 works because `(n-r)!` is exactly the tail of `n!`. Write out `7!` and `4!` by hand and show the cancellation for `P(7,3)`.

In [ ]:
# Your experiments here

---

## Formal Notation

**Factorial** ($n \geq 0$, with $0! = 1$ by convention):
$$n! = n \times (n-1) \times \cdots \times 2 \times 1$$

**Permutations without repetition** ($0 \leq r \leq n$):
$$P(n, r) = \frac{n!}{(n-r)!} = n \times (n-1) \times \cdots \times (n-r+1)$$

The right-hand form is the bounded loop. The fraction form achieves the same thing via tail cancellation.

**Permutations with repetition:**
$$n^r$$

**Combinations without repetition** (also written $\binom{n}{r}$, read "n choose r"):
$$C(n,r) = \binom{n}{r} = \frac{P(n,r)}{r!} = \frac{n!}{r!\,(n-r)!}$$

Note the symmetry: $C(n, r) = C(n, n-r)$. Choosing which $r$ items to include is the same count as choosing which $n-r$ to exclude.

$0! = 1$ is defined so that $P(n,0) = 1$ and $C(n,0) = 1$ are consistent: there is exactly one way to arrange or choose nothing.

---

## Real-World Connection

- **Password security**: the with-repetition formula $n^r$ sizes the brute-force search space. 26 lowercase letters, 8 characters: $26^8 \approx 200$ billion guesses. Adding digits and symbols (72 characters): $72^8 \approx 722$ trillion. Character-set diversity multiplies the exponent's base — length multiplies the exponent.
- **Database query planning**: joining `n` tables has `n!` possible orderings. With 10 tables that is 3.6 million. Query optimisers use heuristics rather than exhaustive search because factorial growth makes enumeration impractical above ~8 tables.
- **Combinations in software testing**: testing all pairs of `n` feature flags is `C(n,2)`. With 20 flags: 190 pairs. Pairwise (2-way) and t-way coverage strategies use combinatorics to find minimal test sets that cover all interactions.
- **Poker probabilities**: a 5-card hand from 52 cards is `C(52,5) = 2,598,960` equally likely hands. Specific hand counts divided by this total give probabilities. Order does not matter — it is a combination, not a permutation.
- **Hyperparameter search**: choosing which 3 hyperparameters to tune from 10 is `C(10,3) = 120`. Choosing which 7 to fix at defaults is `C(10,7) = 120`. The same answer — the $C(n,r) = C(n,n-r)$ symmetry.

---

## Summary

- **Factorial** $n!$ is a countdown multiply loop. It counts complete arrangements of $n$ distinct objects.
- **Permutations without repetition — all objects**: $n!$. Pool drains completely.
- **Permutations without repetition — $r$ from $n$**: the bounded loop $n \times (n-1) \times \cdots \times (n-r+1)$. The formula $\frac{n!}{(n-r)!}$ encodes this as full factorial with tail cancellation. The formula and the loop are the same thing — the formula is code to write, not to run.
- **Permutations with repetition**: $n^r$. Pool never shrinks — independent choice at each position.
- **Combinations without repetition**: $C(n,r) = \frac{P(n,r)}{r!}$. Divide permutations by $r!$ to collapse all orderings of each group into one selection.
- The key relationship: **combinations = permutations / r!**. Permutations count ordered arrangements; combinations count unordered selections.
- $C(n,r) = C(n,n-r)$: choosing $r$ items to include is the same count as choosing $n-r$ items to exclude.